# PIPELINE END-TO-END DỰ ĐOÁN TƯƠNG TÁC THUỐC--THUỐC (DDI)
## BÀI TOÁN HỌC MÁY KHẮC PHỤC HIỆN TƯỢNG RÒ RỈ DỮ LIỆU VÀ KHỞI ĐỘNG LẠNH (COLD-START)
**Học phần:** Học máy (Machine Learning)  
**Nhóm sinh viên thực hiện:** Trần Đặng Vương Quốc Long & Bùi Minh Ảnh -- Lớp KHMT K22A  

---
### QUY TRÌNH KỸ THUẬT END-TO-END:
1. **Giai đoạn 1:** Nạp dữ liệu 10 ma trận tương đồng sinh hóa y tế và ma trận tương tác DDI (nhãn).
2. **Giai đoạn 2:** Lọc nhiễu đặc trưng bằng tỷ lệ Shannon Entropy (ngưỡng $\gamma \ge 0{,}85$).
3. **Giai đoạn 3:** Hợp nhất mạng tương đồng đa nguồn bằng thuật toán **Similarity Network Fusion (SNF)** ($K=20, t=20$).
4. **Giai đoạn 4:** Phân chia dữ liệu theo 4 kịch bản kiểm thử độc lập chống rò rỉ dữ liệu kết hợp lấy mẫu âm cân bằng 1:1.
5. **Giai đoạn 5:** Trích xuất vector đặc trưng biểu diễn quan hệ cặp thuốc $(u, v)$.
6. **Giai đoạn 6:** Huấn luyện 4 mô hình cơ sở (Random Forest, Logistic Regression, KNN, XGBoost) và xây dựng mô hình đề xuất **Soft Voting Ensemble**.
7. **Giai đoạn 7:** Đánh giá đa chiều (ROC-AUC, PR-AUC/AUPR, F1-score) và xuất bảng kết quả thực nghiệm chuẩn mực.

In [ ]:
import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Cố định SEED tái lập kết quả
SEED = 42
np.random.seed(SEED)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Khởi tạo môi trường thành công! SEED =", SEED)

## 1. NẠP DỮ LIỆU ĐẶC TRƯNG & NHÃN TƯƠNG TÁC DDI

In [ ]:
DATA_DIR = 'data'
MATRICES_DIR = os.path.join(DATA_DIR, 'output_matrices')
DDI_MATRIX_PATH = os.path.join(DATA_DIR, 'drug_drug_matrix..csv')

# Đọc ma trận kề DDI
ddi_matrix = pd.read_csv(DDI_MATRIX_PATH, header=None).values
n_drugs = ddi_matrix.shape[0]
print(f"Tổng số thuốc trong hệ thống: {n_drugs}")
print(f"Kích thước ma trận nhãn DDI: {ddi_matrix.shape}")

# 10 kênh đặc trưng tương đồng
matrix_files = {
    'ATC': 'atc_code_jaccard_sim.csv',
    'Chem': 'chem_tanimoto_sim.csv',
    'Enzyme': 'enzyme_jaccard_sim.csv',
    'Indication': 'indication_sim.csv',
    'MeSH': 'mesh_code_jaccard_sim.csv',
    'Offside': 'offside_effect_jaccard_sim.csv',
    'Pathway': 'pathway_jaccard_sim.csv',
    'Side_Effect': 'side_effect_jaccard_sim.csv',
    'Target': 'target_jaccard_sim.csv',
    'Transporter': 'transporter_jaccard_sim.csv',
}

raw_matrices = {}
for name, fname in matrix_files.items():
    fpath = os.path.join(MATRICES_DIR, fname)
    if os.path.exists(fpath):
        mat = pd.read_csv(fpath, header=None).values
        np.fill_diagonal(mat, 1.0)
        mat = np.clip(mat, 0.0, 1.0)
        mat = (mat + mat.T) / 2.0
        raw_matrices[name] = mat
        print(f"Đã nạp kênh {name:12s}: shape {mat.shape}, min={mat.min():.2f}, max={mat.max():.2f}")

## 2. LỌC NHIỄU ĐẶC TRƯNG BẰNG SHANNON ENTROPY ($\gamma \ge 0{,}85$)
Tính tỷ lệ Entropy $\gamma = E / E_{\max}$ cho từng kênh. Những kênh có thông tin nghèo nàn hoặc quá thưa sẽ bị loại bỏ để tránh gây nhiễu cho mô hình.

In [ ]:
def compute_entropy_ratio(matrix, eps=1e-12):
    n = matrix.shape[0]
    triu_vals = matrix[np.triu_indices(n, k=1)]
    total_val = np.sum(triu_vals)
    if total_val <= eps:
        return 0.0
    p = triu_vals / total_val
    p = p[p > eps]
    entropy = -np.sum(p * np.log2(p))
    max_entropy = np.log2(len(triu_vals))
    return entropy / max_entropy if max_entropy > 0 else 0.0

GAMMA_THRESHOLD = 0.85
filtered_matrices = {}
entropy_rows = []

for name, mat in raw_matrices.items():
    gamma = compute_entropy_ratio(mat)
    keep = gamma >= GAMMA_THRESHOLD
    entropy_rows.append({
        'Kênh đặc trưng': name,
        'Tỷ lệ Entropy (gamma)': round(gamma, 4),
        'Ngưỡng chuẩn': GAMMA_THRESHOLD,
        'Quyết định': 'GIỮ LẠI' if keep else 'LOẠI BỎ (Nhiễu)'
    })
    if keep:
        filtered_matrices[name] = mat

if len(filtered_matrices) < 2:
    filtered_matrices = raw_matrices

df_entropy = pd.DataFrame(entropy_rows)
display(df_entropy)

## 3. HỢP NHẤT MẠNG TƯƠNG ĐỒNG BẰNG THUẬT TOÁN SNF ($K=20, t=20$)
Thuật toán **Similarity Network Fusion (SNF)** chuyển đổi các ma trận tương đồng thành đồ thị láng giềng cục bộ $S$ và ma trận chuyển trạng thái toàn cục $P$, sau đó thực hiện $t=20$ vòng lặp khuếch tán để hợp nhất toàn diện 10 góc nhìn sinh học.

In [ ]:
def compute_local_affinity(W, K=20):
    n = W.shape[0]
    S = np.zeros_like(W)
    for i in range(n):
        row = W[i].copy()
        row[i] = -1.0
        top_k = np.argsort(row)[::-1][:K]
        s = np.sum(W[i, top_k])
        if s > 0:
            S[i, top_k] = W[i, top_k] / s
    return S

def compute_global_transition(W):
    n = W.shape[0]
    P = np.zeros_like(W)
    for i in range(n):
        row_sum = np.sum(W[i]) - W[i, i]
        if row_sum > 0:
            P[i, :] = W[i, :] / (2.0 * row_sum)
            P[i, i] = 0.5
        else:
            P[i, i] = 1.0
    return P

def run_snf(sim_dict, K=20, t_iter=20):
    names = list(sim_dict.keys())
    m = len(names)
    mats = [sim_dict[k].copy() for k in names]
    P_list = [compute_global_transition(W) for W in mats]
    S_list = [compute_local_affinity(W, K=K) for W in mats]
    
    for it in range(t_iter):
        P_next = []
        for v in range(m):
            other_sum = np.sum([P_list[j] for j in range(m) if j != v], axis=0)
            P_other_avg = other_sum / (m - 1)
            P_v_next = S_list[v] @ P_other_avg @ (S_list[v].T)
            P_v_next = (P_v_next + P_v_next.T) / 2.0
            np.fill_diagonal(P_v_next, 0.5)
            P_next.append(P_v_next)
        P_list = P_next
        
    P_fused = np.mean(P_list, axis=0)
    P_fused = (P_fused - P_fused.min()) / (P_fused.max() - P_fused.min() + 1e-12)
    np.fill_diagonal(P_fused, 1.0)
    return (P_fused + P_fused.T) / 2.0

print("Đang chạy thuật toán SNF...")
P_fused = run_snf(filtered_matrices, K=20, t_iter=20)
print(f"Hợp nhất SNF hoàn tất! Ma trận P_fused shape: {P_fused.shape}")

## 4. THIẾT LẬP 4 KỊCH BẢN CHỐNG RÒ RỈ DỮ LIỆU & LẤY MẪU ÂM 1:1

In [ ]:
class DataSplitter:
    def __init__(self, interaction_matrix, n_drugs, test_size=0.2, seed=42):
        self.inter = interaction_matrix
        self.n_drugs = n_drugs
        self.test_size = test_size
        self.seed = seed
        
    def split(self, case_id):
        np.random.seed(self.seed)
        if case_id == 1:
            # Case 1: Warm-start (Random pair split)
            idx1, idx2 = np.triu_indices(self.n_drugs, k=1)
            y_all = self.inter[idx1, idx2]
            pos = np.where(y_all == 1)[0]
            neg = np.where(y_all == 0)[0]
            neg_s = np.random.choice(neg, size=len(pos), replace=False)
            all_idx = np.concatenate([pos, neg_s])
            np.random.shuffle(all_idx)
            pairs = np.column_stack([idx1[all_idx], idx2[all_idx]])
            y = y_all[all_idx]
            n_te = int(len(y) * self.test_size)
            perm = np.random.permutation(len(y))
            return pairs[perm[n_te:]], pairs[perm[:n_te]], y[perm[n_te:]], y[perm[:n_te]]
            
        # Case 2, 3, 4: Drug-disjoint node split
        drugs = np.arange(self.n_drugs)
        np.random.shuffle(drugs)
        n_te = int(self.n_drugs * self.test_size)
        te_drugs, tr_drugs = drugs[:n_te], drugs[n_te:]
        
        # Train (Seen x Seen)
        t1, t2 = np.triu_indices(len(tr_drugs), k=1)
        p_tr1, p_tr2 = tr_drugs[t1], tr_drugs[t2]
        y_tr = self.inter[p_tr1, p_tr2]
        pos_tr = np.where(y_tr == 1)[0]
        neg_tr = np.where(y_tr == 0)[0]
        neg_tr_s = np.random.choice(neg_tr, size=len(pos_tr), replace=False)
        b_tr = np.concatenate([pos_tr, neg_tr_s])
        np.random.shuffle(b_tr)
        pairs_train = np.column_stack([p_tr1[b_tr], p_tr2[b_tr]])
        y_train = y_tr[b_tr]
        
        if case_id == 2:
            # New x New
            e1, e2 = np.triu_indices(len(te_drugs), k=1)
            p_te1, p_te2 = te_drugs[e1], te_drugs[e2]
            y_te = self.inter[p_te1, p_te2]
            pos_te = np.where(y_te == 1)[0]
            neg_te = np.where(y_te == 0)[0]
            neg_te_s = np.random.choice(neg_te, size=len(pos_te), replace=False)
            b_te = np.concatenate([pos_te, neg_te_s])
            pairs_test = np.column_stack([p_te1[b_te], p_te2[b_te]])
            y_test = y_te[b_te]
        elif case_id == 3:
            # Old x New
            m_s, m_u = np.meshgrid(tr_drugs, te_drugs, indexing='ij')
            s_f, u_f = m_s.flatten(), m_u.flatten()
            y_c = self.inter[s_f, u_f]
            pos_c = np.where(y_c == 1)[0]
            neg_c = np.where(y_c == 0)[0]
            sz = min(len(pos_c), len(neg_c))
            b_c = np.concatenate([np.random.choice(pos_c, sz, False), np.random.choice(neg_c, sz, False)])
            pairs_test = np.column_stack([s_f[b_c], u_f[b_c]])
            y_test = y_c[b_c]
        elif case_id == 4:
            # Mix (Case 2 + Case 3)
            e1, e2 = np.triu_indices(len(te_drugs), k=1)
            y_te = self.inter[te_drugs[e1], te_drugs[e2]]
            pos_te, neg_te = np.where(y_te==1)[0], np.where(y_te==0)[0]
            b_te = np.concatenate([pos_te, np.random.choice(neg_te, len(pos_te), False)])
            p2 = np.column_stack([te_drugs[e1][b_te], te_drugs[e2][b_te]])
            y2 = y_te[b_te]
            
            m_s, m_u = np.meshgrid(tr_drugs, te_drugs, indexing='ij')
            s_f, u_f = m_s.flatten(), m_u.flatten()
            y_c = self.inter[s_f, u_f]
            pos_c, neg_c = np.where(y_c==1)[0], np.where(y_c==0)[0]
            sz = min(len(pos_c), len(neg_c))
            b_c = np.concatenate([np.random.choice(pos_c, sz, False), np.random.choice(neg_c, sz, False)])
            p3 = np.column_stack([s_f[b_c], u_f[b_c]])
            y3 = y_c[b_c]
            
            pairs_test = np.concatenate([p2, p3])
            y_test = np.concatenate([y2, y3])
            perm = np.random.permutation(len(y_test))
            pairs_test, y_test = pairs_test[perm], y_test[perm]
            
        return pairs_train, pairs_test, y_train, y_test

splitter = DataSplitter(ddi_matrix, n_drugs, test_size=0.2, seed=SEED)

## 5. TRÍCH XUẤT ĐẶC TRƯNG CẶP THUỐC $(u, v)$

In [ ]:
def extract_features(pairs, fused_mat, sim_dict):
    u, v = pairs[:, 0], pairs[:, 1]
    feats = [fused_mat[u, v].reshape(-1, 1)]
    for name, mat in sim_dict.items():
        feats.append(mat[u, v].reshape(-1, 1))
    
    u_vecs, v_vecs = fused_mat[u, :], fused_mat[v, :]
    cos_sim = np.sum(u_vecs * v_vecs, axis=1) / ((np.linalg.norm(u_vecs, axis=1)+1e-12)*(np.linalg.norm(v_vecs, axis=1)+1e-12))
    feats.append(cos_sim.reshape(-1, 1))
    
    had = u_vecs * v_vecs
    feats.extend([np.mean(had, axis=1, keepdims=True),
                  np.std(had, axis=1, keepdims=True),
                  np.max(had, axis=1, keepdims=True)])
    return np.hstack(feats)

## 6. HUẤN LUYỆN CÁC MÔ HÌNH HỌC MÁY & SOFT VOTING ENSEMBLE
Đánh giá trên 4 kịch bản bằng 3 chỉ số: **ROC-AUC**, **PR-AUC (AUPR)** và **F1-score**.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    HAS_XGB = False

scenarios = [
    (1, "Case 1 (Warm-start)"),
    (2, "Case 2 (Strict Cold-start)"),
    (3, "Case 3 (Semi Cold-start)"),
    (4, "Case 4 (Mixed Scenario)")
]

results = []

for cid, cname in scenarios:
    print(f"\nĐang thực nghiệm {cname}...")
    p_tr, p_te, y_tr, y_te = splitter.split(cid)
    X_tr = extract_features(p_tr, P_fused, filtered_matrices)
    X_te = extract_features(p_te, P_fused, filtered_matrices)
    
    models = {
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=SEED),
        'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        'XGBoost': XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=SEED, eval_metric='logloss', n_jobs=-1) if HAS_XGB else GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=SEED)
    }
    
    preds = {}
    for mname, model in models.items():
        model.fit(X_tr, y_tr)
        prob = model.predict_proba(X_te)[:, 1]
        preds[mname] = prob
        r_auc = roc_auc_score(y_te, prob)
        p, r, _ = precision_recall_curve(y_te, prob)
        pr_auc = auc(r, p)
        f1 = f1_score(y_te, (prob >= 0.5).astype(int), zero_division=0)
        results.append({
            'Kịch bản': cname,
            'Mô hình': mname,
            'ROC-AUC': round(r_auc, 4),
            'PR-AUC': round(pr_auc, 4),
            'F1-score': round(f1, 4)
        })
        
    # Soft Voting Ensemble
    ens_prob = np.mean(list(preds.values()), axis=0)
    ens_r_auc = roc_auc_score(y_te, ens_prob)
    p_e, r_e, _ = precision_recall_curve(y_te, ens_prob)
    ens_pr_auc = auc(r_e, p_e)
    ens_f1 = f1_score(y_te, (ens_prob >= 0.5).astype(int), zero_division=0)
    results.append({
        'Kịch bản': cname,
        'Mô hình': 'Soft Voting Ensemble (Đề xuất)',
        'ROC-AUC': round(ens_r_auc, 4),
        'PR-AUC': round(ens_pr_auc, 4),
        'F1-score': round(ens_f1, 4)
    })

df_res = pd.DataFrame(results)
display(df_res)